# Notebook A -- Phase 1 (paper-faithful), ALL sessions, 6 classes, Leave-One-Speaker-Out

**Algorithm 1, step 1 of the paper:** fine-tune the Wav2Vec 2.0 temporal module on the **clear (SA) samples only** -> `M1p`.
This notebook does it once **per held-out speaker**, so that no checkpoint ever sees the test speaker (the paper evaluates with LOSO, Sec. V-A).

| | Paper | Your old notebook | This notebook |
|---|---|---|---|
| Training data | clear SA samples | SA | SA of the 9 training speakers |
| Learning rate | fixed 1e-5 | backbone 2e-6 / head 5e-5, ReduceLROnPlateau | **fixed 1e-5**, one param group |
| Transformer layers | not frozen (not mentioned) | 10 of 12 frozen, then gradual unfreezing | **all trainable** (`FREEZE_TRANSFORMER_LAYERS = 0`) |
| Class weights | inverse class frequency | none | **inverse frequency**, from training speakers only (capped like your Phase 3) |
| Masking | HF wav2vec2-base-960h default | default | default (printed as a check) |
| Evaluation | LOSO | on the training data (95% = memorisation) | **held-out speaker** |
| Classes | 4 | 6 | 6 |

Output: one `phase1_<speaker>.pt` per fold in `/kaggle/working/phase1_loso/` (+ `phase1_loso_results.json`). The Phase 2/3 LOSO notebook loads the checkpoint of the matching fold.

## 1. Install dependencies

In [1]:
# On Kaggle's GPU T4x2 image, torch/torchaudio are already installed and matched
# to the CUDA driver -- reinstalling them with pip can silently swap in a CPU-only
# or mismatched build. Only install/upgrade what's actually missing.
!pip install -q -U transformers
!pip install -q ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 39.8 MB/s eta 0:00:00


In [2]:
import torch
import transformers
import torchaudio

print("PyTorch Version:", torch.__version__)
print("Transformers Version:", transformers.__version__)
print("Is CUDA (GPU) available?", torch.cuda.is_available())


PyTorch Version: 2.11.0+cu128
Transformers Version: 5.19.0
Is CUDA (GPU) available? True


## 2. Kaggle paths & device setup

In [3]:
import os, glob, torch

# ---- Device & GPU info ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print("Using device:", device, "| GPUs visible:", n_gpus)
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

def maybe_parallel(model):
    """Wrap in DataParallel for the forward pass only; always optimise/save the ORIGINAL model."""
    return torch.nn.DataParallel(model) if n_gpus > 1 else model

# ---- Locate the preprocessed .pt file(s) ----------------------------------------------------
# Put your file(s) here, e.g. ["/kaggle/input/<your-dataset>/processed_all_sessions.pt"].
# Leave EMPTY to auto-discover (works if exactly one candidate .pt exists under /kaggle/input).
PT_FILES = ["/kaggle/input/datasets/bdivyabalamurugan/preprocess-session-1-to-5/processed/processed_first_all.pt"]

INPUT_ROOT = "/kaggle/input"
if not PT_FILES:
    cands = [p for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*.pt"), recursive=True)
             if not any(w in os.path.basename(p).lower() for w in ("phase1", "ckpt", "checkpoint", "resume"))]
    if len(cands) == 1:
        PT_FILES = cands
    else:
        print(f"Auto-discovery found {len(cands)} candidate .pt file(s):")
        for p in cands: print("   ", p)
        raise FileNotFoundError("Set PT_FILES above to the right file path(s) (a list), then re-run this cell. "
                                "If the list above is empty, attach your preprocessed dataset via '+ Add Input'.")
for p in PT_FILES:
    assert os.path.exists(p), f"missing: {p}"
print("PT_FILES ->", PT_FILES)

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR ->", OUTPUT_DIR)

Using device: cuda | GPUs visible: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PT_FILES -> ['/kaggle/input/datasets/bdivyabalamurugan/preprocess-session-1-to-5/processed/processed_first_all.pt']
OUTPUT_DIR -> /kaggle/working


## 3. Temporal Module (Wav2Vec 2.0) -- Section IV-C

In [4]:
"""
================================================================================
Temporal Module (Wav2Vec 2.0) -- Section IV-C / Fig. 4 of the paper
================================================================================
INPUT
-----
x_i : raw 16kHz waveform, shape (T,) -- T = number of RAW AUDIO SAMPLES, which
also varies between utterances (e.g. your own samples: 24000, 53360, 112000
samples for different clip lengths). No fixed input size here either, for
the same underlying reason as the CNN branch: this ends in a POOLING step
that collapses variable length down to a fixed-size vector.

ARCHITECTURE (matches Eq. 24-28 + Fig. 4)
------------------------------------------
Wav2Vec 2.0 itself already contains:
  a) A stack of 1D convolutions (its own internal "feature encoder") that
     turns the raw waveform into a sequence of frames:
         G ∈ R^{T' x d1}   (T' = number of frames, varies with input length;
                             d1 = 512 for wav2vec2-base)
  b) A linear projection into the transformer's hidden size:
         G' = Linear(G) ∈ R^{T' x d2}     (d2 = 768 for wav2vec2-base)   -- Eq. 24
  c) An optional MASKING step during fine-tuning (randomly masks some frames,
     a regularization trick borrowed from masked-language-modeling) -- Eq. 25.
     This is a BUILT-IN feature of HuggingFace's Wav2Vec2Model, controlled by
     config.mask_time_prob / config.mask_time_length, and (per Table I in the
     paper) uses the HuggingFace wav2vec2-base-960h model's DEFAULT masking
     config -- so we don't hand-implement masking, we just use the pretrained
     model as-is with masking enabled during training.
  d) The Transformer encoder stack (also built into Wav2Vec2Model):
         F2 = Transformer(F1) ∈ R^{T' x d2}                              -- Eq. 26

WHAT WE ADD ON TOP (this is the paper's own custom aggregation, Eq. 27-28):
Wav2Vec 2.0's native output F2 is still PER-FRAME (T' x d2) -- one vector
per time step, and T' varies between utterances. To get one FIXED-SIZE
vector per utterance (needed for fusion with the CNN's Xs), the paper adds:
  e) A learned linear layer that turns each frame's d2-dim vector into a
     single attention-style weight:
         F2' = Linear(F2) ∈ R^{1 x T'}                                    -- Eq. 27
     (we apply softmax over T' here, turning F2' into proper attention
      weights that sum to 1 -- a standard, reasonable choice consistent
      with the paper's "weighted aggregation" description, though the paper
      doesn't spell out softmax explicitly)
  f) Multiply those weights by F2 and sum over time (matrix multiplication):
         Ft = F2' ⊗ F2 ∈ R^{1 x d2}                                       -- Eq. 28
     This is a weighted average over ALL T' frames -- so REGARDLESS of how
     many frames T' there were, Ft always comes out as a fixed d2-dim vector.

Ft is the fixed-size temporal embedding fed into the Multi-Level Fusion
Module (Sec. IV-D) alongside the spatial embedding Xs from the CNN.

DEPENDENCY NOTE
---------------
Requires the `transformers` library (HuggingFace):
    pip install transformers
Uses the pretrained "facebook/wav2vec2-base-960h" checkpoint, matching the
paper's Table I description of the masking configuration.

*** NEW: `set_unfrozen_layers()` (added for gradual unfreezing) ***
Training was learning slowly because 10/12 transformer layers were frozen
for the ENTIRE run (a safe-but-slow fix for earlier NaN instability). This
method lets the training loop progressively open up more layers as training
stabilizes, instead of being stuck at 10/12 frozen forever. See the Phase 1
and Phase 3 training-loop cells below for how it's used.
================================================================================
"""

import torch
import torch.nn as nn
from transformers import Wav2Vec2Model, Wav2Vec2Config


class TemporalWav2Vec2(nn.Module):
    """
    Full Temporal Module, Section IV-C. Input: (B, T) raw 16kHz waveform.
    Output: (B, 768) -- ALWAYS 768-dim (wav2vec2-base's hidden size),
    regardless of T (the number of raw audio samples in the input).
    """

    def __init__(self, pretrained_name: str = "/kaggle/input/datasets/bdivyabalamurugan/wav2vec-model/wav2vec2_stored",
                 freeze_feature_encoder: bool = True,
                 freeze_transformer_layers: int = 10):
        """
        freeze_transformer_layers: how many of wav2vec2-base's 12 transformer
        encoder layers to FREEZE (starting from layer 0, the earliest/most
        general ones), leaving the remaining top layers + our pooling head
        trainable. This is the standard, well-documented fix for wav2vec2-base
        fine-tuning instability (NaN loss/gradients): fully unfreezing all 12
        transformer layers at once, on top of a randomly-initialized head,
        lets large early gradients from the head propagate all the way back
        through the pretrained weights and destabilize them. Only letting the
        last ~2 layers adapt keeps the bulk of the pretrained representation
        intact while still allowing task-specific adaptation near the output.
        Set to 0 to fine-tune all 12 layers (the old, unstable behavior).
        """
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(pretrained_name)
        hidden_size = self.wav2vec2.config.hidden_size  # 768 for wav2vec2-base

        if freeze_feature_encoder:
            # Common practice: freeze the low-level CNN feature extractor
            # (already well pretrained on huge unlabeled speech corpora),
            # and only fine-tune the transformer layers + our new pooling head.
            self.wav2vec2.feature_extractor._freeze_parameters()

        self.set_unfrozen_layers(freeze_transformer_layers)

        # *** FIX: freeze masked_spec_embed ***
        # This is Wav2Vec2's single learned "mask token" vector (substituted
        # in for masked time-steps during spec-augment-style training). It's
        # a standalone top-level Parameter, NOT part of encoder.layers, so the
        # freezing loop above never touched it -- it stayed fully trainable
        # and, over 8 epochs, its magnitude ran away to ~1.89e38 (right at
        # float32's ceiling of ~3.4e38, one step from overflowing to inf).
        # A near-infinite value substituted into the hidden-state sequence
        # for every masked frame is very likely what's been driving the
        # batch-wide NaNs all along. There's no real need to fine-tune this
        # token for our task, so we freeze it at its pretrained value.
        if hasattr(self.wav2vec2, "masked_spec_embed"):
            mse = self.wav2vec2.masked_spec_embed
            # *** THE ACTUAL FIX ***
            # Freezing this parameter (below) only stops it from getting WORSE
            # during fine-tuning -- it does nothing if the value was already
            # broken the moment the checkpoint loaded. And that's exactly
            # what's happening: HuggingFace initializes masked_spec_embed with
            # Uniform(0, 1) -- a healthy value should never be anywhere near
            # 1e38. Finding it already at ~1.89e38 BEFORE any training step
            # means this specific locally-stored checkpoint file
            # (wav2vec2_stored) has this one parameter corrupted at the
            # source, unrelated to anything in this notebook. Freezing a
            # broken value just keeps it broken forever, which is why the
            # skip-count kept climbing at the same rate even after we froze
            # it last time. The real fix: detect this and RESET it to a sane
            # value (matching HF's own default init) before freezing it.
            if not torch.isfinite(mse).all() or mse.abs().max().item() > 10.0:
                print(f"[TemporalWav2Vec2] WARNING: masked_spec_embed loaded "
                      f"from the checkpoint is already corrupted "
                      f"(max abs={mse.abs().max().item():.3e}), BEFORE any "
                      f"fine-tuning -- this is a broken checkpoint file, not "
                      f"a training issue. Re-initializing it to HF's own "
                      f"default Uniform(0, 1) range instead of using the "
                      f"broken saved value.")
                with torch.no_grad():
                    mse.uniform_(0.0, 1.0)
            mse.requires_grad = False
            print("[TemporalWav2Vec2] Froze masked_spec_embed (mask-token "
                  "vector) to prevent it from diverging during fine-tuning.")

        # Eq. 27: per-frame -> single attention-style scalar weight
        self.attn_proj = nn.Linear(hidden_size, 1)

    def set_unfrozen_layers(self, freeze_transformer_layers: int) -> int:
        """*** NEW: supports GRADUAL UNFREEZING ***
        Freezes the first `freeze_transformer_layers` encoder layers and
        unfreezes the rest. Called once from __init__ (the original static
        behavior), and can also be called again LATER, mid-training, by the
        training loop to progressively open up more of the backbone once
        the loss has stabilized -- instead of leaving 10/12 layers frozen
        for the entire run, which is safe but needlessly slow.
        Returns the actual number of frozen layers (clamped to [0, n_layers]).
        """
        n_layers = len(self.wav2vec2.encoder.layers)   # 12 for wav2vec2-base
        n_freeze = max(0, min(freeze_transformer_layers, n_layers))
        for i, layer in enumerate(self.wav2vec2.encoder.layers):
            requires_grad = i >= n_freeze
            for p in layer.parameters():
                p.requires_grad = requires_grad
        print(f"[TemporalWav2Vec2] Froze transformer layers 0-{n_freeze-1} "
              f"of {n_layers}; layers {n_freeze}-{n_layers-1} + pooling head "
              f"remain trainable.")
        return n_freeze

    @staticmethod
    def _normalize_waveform(waveform: torch.Tensor, attention_mask: torch.Tensor = None,
                             eps: float = 1e-7) -> torch.Tensor:
        """*** THE REAL, COMPLETE FIX FOR THE NaN BUG ***
        facebook/wav2vec2-base-960h expects ZERO-MEAN, UNIT-VARIANCE normalized
        audio. The FIRST fix (plain mean()/std() over the whole array) was
        INCOMPLETE: when a batch mixes clips of different real lengths, shorter
        clips get zero-padded to match the longest one in the batch -- e.g. a
        1.5s clip (24000 samples) padded to match a 7s clip (112000 samples)
        has 88000 fake zero samples (78% of its length!). Computing mean/std
        over THAT ENTIRE padded array lets the huge zero-block distort the
        statistics -- std becomes artificially tiny, so dividing by it blows
        the real audio values up to extreme magnitudes -> NaN inside
        Wav2Vec2's internal layers. THIS is what was still causing
        "RuntimeError: NaN/Inf detected..." during Phase 1 training.

        FIX: when an attention_mask is given (1=real audio, 0=padding), compute
        mean/std using ONLY the real (non-padded) positions per sample, so
        padding never distorts the statistics. Falls back to plain mean/std
        when no mask is given (single, un-padded samples -- mathematically
        identical to masked in that case anyway)."""
        # MIN_STD floors the denominator so near-silent/very-low-energy clips don't get
        # divided by a tiny std and amplified into an unrealistic dynamic range that
        # blows up inside Wav2Vec2's pretrained conv stack (the real cause of the
        # "NaN/Inf detected" error even after the padding-aware fix below). The final
        # clamp is a second, independent safety net on the output magnitude itself.
        MIN_STD = 1e-3
        CLAMP_RANGE = 10.0

        # *** ADDITIONAL FIX: sanitize the RAW waveform itself before computing any
        # statistics. The MIN_STD floor + output clamp above only protect against a
        # tiny-but-finite std (e.g. near-silent clips). They do NOT protect against
        # NaN/Inf values that are already present in the input tensor (e.g. from a
        # corrupted source audio file, a bad resample, or a division-by-zero further
        # upstream in preprocessing). If even one NaN/Inf sneaks into `waveform`,
        # mean()/std() over that sample become NaN too, and MIN_STD.clamp() cannot
        # fix a NaN (clamp does not touch NaN values) -- so the NaN silently survives
        # into Wav2Vec2 and triggers the "NaN/Inf detected" crash regardless of the
        # floor/clamp below. Replacing NaN/Inf with 0 here removes the problem at its
        # source, before any statistic is computed from it.
        if not torch.isfinite(waveform).all():
            waveform = torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)

        if attention_mask is None:
            mean = waveform.mean(dim=-1, keepdim=True)
            std = waveform.std(dim=-1, keepdim=True).clamp(min=MIN_STD)
            normalized = (waveform - mean) / std
            return normalized.clamp(-CLAMP_RANGE, CLAMP_RANGE)

        mask = attention_mask.to(waveform.dtype)                      # (B, T)
        lengths = mask.sum(dim=-1, keepdim=True).clamp(min=1.0)         # (B, 1) real length per sample
        mean = (waveform * mask).sum(dim=-1, keepdim=True) / lengths      # masked mean
        var = ((waveform - mean) ** 2 * mask).sum(dim=-1, keepdim=True) / lengths
        std = torch.sqrt(var.clamp(min=0.0)).clamp(min=MIN_STD)
        normalized = (waveform - mean) / std
        normalized = normalized.clamp(-CLAMP_RANGE, CLAMP_RANGE)
        return normalized * mask   # keep padded positions at exactly 0 after normalizing

    def forward(self, waveform: torch.Tensor,
                attention_mask: torch.Tensor = None) -> torch.Tensor:
        # waveform: (B, T) raw 16kHz audio -> mask-aware normalize BEFORE Wav2Vec2 sees it
        waveform = self._normalize_waveform(waveform, attention_mask)

        outputs = self.wav2vec2(waveform, attention_mask=attention_mask)
        F2 = outputs.last_hidden_state          # (B, T', 768) -- Eq. 26, T' varies with T

        # *** SELF-HEALING FIX (replaces the old hard-crash check) ***
        # We already confirmed (via the diagnostic+filter cell) that the RAW
        # waveforms going in are all finite -- no NaN/Inf, no near-silent clips.
        # So a NaN appearing HERE, in Wav2Vec2's own output, is NOT a bad-data
        # problem -- it's numerical instability inside wav2vec2-base's transformer
        # layers while fine-tuning (a well-known issue with this checkpoint, since
        # it uses post-norm rather than the more stable pre-norm variant used in
        # wav2vec2-large-robust/xlsr). We can't reliably prevent every occurrence
        # of this with a normalization tweak alone, so instead of crashing the
        # whole training run over ONE bad sample in a batch, we:
        #   1. find exactly which sample(s) in this batch went non-finite,
        #   2. zero out just those rows (so they contribute ~nothing and don't
        #      poison the rest of the batch via the pooled attention step below),
        #   3. print a warning so you can see how often / which ones this hits.
        # Training simply continues instead of dying.
        sample_finite = torch.isfinite(F2).flatten(1).all(dim=1)   # (B,) bool
        if not sample_finite.all():
            bad_idx = (~sample_finite).nonzero(as_tuple=True)[0].tolist()
            print(f"[TemporalWav2Vec2] WARNING: non-finite output for local batch "
                  f"indices {bad_idx} (of {F2.shape[0]}) -- zeroing these rows and "
                  f"continuing. If this keeps happening on the SAME utt_id, that "
                  f"clip should be excluded from training; if it's scattered/random, "
                  f"it's fine-tuning instability -- consider lowering the Phase 1 "
                  f"learning rate.")
            F2 = F2.clone()
            F2[~sample_finite] = 0.0

        # Eq. 27: per-frame scalar -> softmax over time -> attention weights
        attn_logits = self.attn_proj(F2)         # (B, T', 1)
        attn_logits = attn_logits.squeeze(-1)    # (B, T')
        if attention_mask is not None:
            # Wav2Vec2 downsamples the mask internally; use its helper to
            # get the mask at the RIGHT resolution for F2 before softmax,
            # so padded frames get (effectively) zero weight.
            feat_mask = self.wav2vec2._get_feature_vector_attention_mask(
                F2.shape[1], attention_mask
            )
            # *** THE ACTUAL ROOT CAUSE OF THE RECURRING NaN GRADIENTS ***
            # If ANY sample in the batch has zero valid (unpadded) frames at
            # this downsampled resolution, masking its entire row to literal
            # -inf makes softmax compute (-inf) - max(-inf) = NaN for every
            # entry in that row (an indeterminate inf-minus-inf, NOT 0). That
            # one NaN row then poisons attn_proj's gradient for the WHOLE
            # BATCH, since attn_proj is one shared Linear layer -- which is
            # exactly why lowering the LR / freezing more transformer layers
            # never fixed this: the instability was never in wav2vec2's own
            # weights, it was here. Using a large FINITE negative sentinel
            # instead of -inf avoids the NaN entirely: softmax subtracts the
            # row max (which is also -1e9 in a fully-masked row), giving 0
            # for every entry -> a harmless uniform distribution instead of
            # NaN. For normal rows (at least one real frame), a padded
            # frame's weight is still ~0 after softmax, same as before.
            attn_logits = attn_logits.masked_fill(feat_mask == 0, -1e9)
            n_fully_masked = (feat_mask.sum(dim=1) == 0).sum().item()
            if n_fully_masked > 0:
                print(f"[TemporalWav2Vec2] NOTE: {n_fully_masked} sample(s) in "
                      f"this batch had ZERO valid frames after downsampling -- "
                      f"falling back to uniform attention for them. This usually "
                      f"means a clip's real (unpadded) length is too short even "
                      f"after zero-padding; worth checking your preprocessing's "
                      f"minimum-length floor if this prints often.")
        attn_weights = torch.softmax(attn_logits, dim=-1)   # (B, T'), sums to 1

        # Eq. 28: Ft = F2' ⊗ F2  (weighted sum over time -> fixed-size vector)
        Ft = torch.bmm(attn_weights.unsqueeze(1), F2)   # (B, 1, 768)
        Ft = Ft.squeeze(1)                               # (B, 768) <- FIXED SIZE
        return Ft


## 4. Load ALL samples (6 classes), clean, and find each sample's speaker

In [5]:
import re, json, gc, time, math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter

EMOTION_CLASSES = ["anger", "happiness", "sadness", "neutral", "frustration", "fear"]   # 6 classes
NUM_CLASSES = len(EMOTION_CLASSES)

all_samples = []
for pth in PT_FILES:
    blob = torch.load(pth, weights_only=False)
    smp = blob["samples"]
    tag = os.path.splitext(os.path.basename(pth))[0]
    for s in smp:
        s["orig_utt_id"] = s["utt_id"]
        if len(PT_FILES) > 1:                      # ids like utt_000027 repeat across files -> make unique
            s["utt_id"] = f"{tag}/{s['utt_id']}"
    all_samples += smp
    print(f"{os.path.basename(pth)}: {len(smp)} samples | blob keys: {list(blob.keys())}")

print("fields in one sample:", sorted(all_samples[0].keys()))
n_lab = all_samples[0]["hard_label"].numel()
assert n_lab == NUM_CLASSES, (f"Your .pt has {n_lab}-dim labels but this notebook expects {NUM_CLASSES} classes in the order "
                              f"{EMOTION_CLASSES}. Re-run preprocessing with 6 classes.")

class PTFileDataset(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        label = s["hard_label"] if s["split"] == "SA" else s["soft_label"]     # SA: one-hot (Eq.1), SB: soft (Eq.5)
        return {"waveform": s["waveform"], "fbank": s["fbank"], "label": label, "split": s["split"], "utt_id": s["utt_id"]}

def collate_fn_training(batch):
    max_wav = max(b["waveform"].shape[-1] for b in batch)
    max_frames = max(b["fbank"].shape[1] for b in batch)
    wavs, fbs, labels, splits, masks, ids = [], [], [], [], [], []
    for b in batch:
        n = b["waveform"].shape[-1]
        wavs.append(F.pad(b["waveform"], (0, max_wav - n)))
        m = torch.zeros(max_wav, dtype=torch.long); m[:n] = 1; masks.append(m)
        fbs.append(F.pad(b["fbank"], (0, 0, 0, max_frames - b["fbank"].shape[1])))
        labels.append(b["label"]); splits.append(b["split"]); ids.append(b["utt_id"])
    return {"waveform": torch.stack(wavs), "fbank": torch.stack(fbs), "label": torch.stack(labels),
            "split": splits, "attention_mask": torch.stack(masks), "utt_id": ids}

n_sa = sum(s["split"] == "SA" for s in all_samples)
print(f"\nTotal samples: {len(all_samples)}  (SA clear: {n_sa}, SB ambiguous: {len(all_samples) - n_sa})")

processed_first_all.pt: 11366 samples | blob keys: ['samples', 'meta']
fields in one sample: ['fbank', 'hard_label', 'is_ambiguous', 'orig_utt_id', 'session', 'soft_label', 'speaker', 'split', 'utt_id', 'waveform']

Total samples: 11366  (SA clear: 8515, SB ambiguous: 2851)


In [6]:
# ---- Remove unusable clips (same filter as your old notebook, now on the merged list) ----
APPLY_BAD_CLIP_FILTER = True
MIN_STD_THRESHOLD = 1e-3

if APPLY_BAD_CLIP_FILTER:
    bad = []
    for s in all_samples:
        w = s["waveform"].float()
        if (not torch.isfinite(w).all()) or w.std().item() < MIN_STD_THRESHOLD:
            bad.append(s["utt_id"])
    bad = set(bad)
    nb_before = len(all_samples)
    all_samples = [s for s in all_samples if s["utt_id"] not in bad]
    print(f"Dropped {len(bad)} NaN/Inf or near-silent clips ({nb_before} -> {len(all_samples)}). First few: {sorted(bad)[:8]}")

Dropped 1074 NaN/Inf or near-silent clips (11366 -> 10292). First few: ['utt_000002_seg2', 'utt_000007', 'utt_000018_seg1', 'utt_000021', 'utt_000046_seg1', 'utt_000062', 'utt_000068_seg1', 'utt_000071_seg1']


In [7]:
# ---- Which speaker does each sample belong to? ----
# Your utt_ids look like 'utt_000027' (anonymous), so the speaker has to come from another field.
# This looks, in order, at: an explicit speaker field -> an IEMOCAP-style name inside ANY string field of the sample
# (e.g. a wav path or original utterance id) -> a session field (then folds are leave-one-SESSION-out).
SPEAKER_KEYS = ["speaker", "speaker_id", "spk", "spk_id"]
SESSION_KEYS = ["session", "session_id", "sess"]
_IEMO_RE = re.compile(r"Ses(\d{2})[FM]_.*?_([FM])\d+")

def get_group(s):
    for k in SPEAKER_KEYS:
        if s.get(k) is not None:
            return str(s[k])
    for k, v in s.items():
        if isinstance(v, str):
            m = _IEMO_RE.search(v)
            if m:
                return f"S{m.group(1)}{m.group(2)}"
    for k in SESSION_KEYS:
        if s.get(k) is not None:
            return f"Session{s[k]}"
    return None

for s in all_samples:
    s["group"] = get_group(s)

n_missing = sum(s["group"] is None for s in all_samples)
if n_missing:
    ex = next(s for s in all_samples if s["group"] is None)
    print("A sample with no speaker info -> non-tensor fields:", {k: v for k, v in ex.items() if not torch.is_tensor(v)})
    raise RuntimeError(
        f"{n_missing}/{len(all_samples)} samples carry no speaker/session information, so Leave-One-Speaker-Out cannot be built.\n"
        f"Fix: re-run your preprocessing so each sample dict also stores the speaker (e.g. s['speaker']='S01F', taken from the "
        f"IEMOCAP file name Ses01F_impro01_F000 -> session 01 + the gender letter after the last underscore), "
        f"then rebuild the .pt. Or edit get_group() above if the information is stored under another field name.")

GROUPS = sorted({s["group"] for s in all_samples})
kind = "SESSION (leave-one-session-out)" if all(g.startswith("Session") for g in GROUPS) else "SPEAKER (LOSO)"
print(f"\nFold unit: {kind} -> {len(GROUPS)} folds: {GROUPS}")
if len(GROUPS) != 10 and kind.startswith("SPEAKER"):
    print(f"!! IEMOCAP has 10 speakers but {len(GROUPS)} were found -- is the .pt really all 5 sessions?")

print(f"\n{'group':10s}{'SA':>6s}{'SB':>6s}   " + "  ".join(f"{c[:5]:>5s}" for c in EMOTION_CLASSES) + "   (SA class counts)")
for g in GROUPS:
    ss = [s for s in all_samples if s["group"] == g]
    sa = [s for s in ss if s["split"] == "SA"]
    cnt = Counter(s["hard_label"].argmax().item() for s in sa)
    print(f"{g:10s}{len(sa):6d}{len(ss)-len(sa):6d}   " + "  ".join(f"{cnt.get(k, 0):5d}" for k in range(NUM_CLASSES)))


Fold unit: SPEAKER (LOSO) -> 10 folds: ['Ses01F', 'Ses01M', 'Ses02F', 'Ses02M', 'Ses03F', 'Ses03M', 'Ses04F', 'Ses04M', 'Ses05F', 'Ses05M']

group         SA    SB   anger  happi  sadne  neutr  frust   fear   (SA class counts)
Ses01F       713   231     181    174     81    165    106      6
Ses01M       832   210      97    156    120    260    189     10
Ses02F       696   196      79    166     93    158    191      9
Ses02M       808   209      81    141     90    286    203      7
Ses03F       626   323      94    126     82    124    196      4
Ses03M       784   252     171    135     84    215    179      0
Ses04F       746   257     226    154     64     86    212      4
Ses04M       875   347     131     62     96    221    360      5
Ses05F       710   239      85    130     92    218    179      6
Ses05M       929   309     107    231    124    174    285      8


## 5. Phase 1 per fold -- paper settings
Fixed LR 1e-5, Adam, inverse-frequency class weights, HF default masking, no layer freezing (only Wav2Vec2's CNN feature encoder stays frozen, which is the standard HF recipe).
The held-out speaker is **never** used for training, class weights, or choosing the epoch: the *final-epoch* model is the one that is saved and scored.

> **Compute:** every fold fine-tunes the whole Wav2Vec2 model on ~9/10 of the clear data, so 10 folds is a lot of GPU time. Check the time of the first epochs, then pick `EPOCHS_PHASE1` / `FOLDS_TO_RUN`. Each finished fold is saved; just re-run the cell in a new session and it continues.

In [8]:
# ============================================================================
# Phase 1 config  (paper: fixed LR 1e-5; class-weighted CE; HF default masking)
# ============================================================================
PHASE1_LR = 1e-5                 # paper: fixed learning rate 1e-5 (no scheduler, no differential LR)
EPOCHS_PHASE1 = 10               # per fold -- NOT given in the text I could read; tune to your GPU budget
BATCH_SIZE = 8
FREEZE_TRANSFORMER_LAYERS = 0    # 0 = fine-tune all 12 layers (paper). If you hit OOM/NaN: try 6, then 10.
MAX_CLASS_WEIGHT = 8.0           # same cap as your Phase 3; set None for the raw inverse-frequency weights
CHECKPOINT_EVERY = 5             # resume point inside a fold
EVAL_EVERY = 5                   # monitor-only test accuracy during training (0 = off); never used to select anything

FOLDS_TO_RUN = None              # None = all, or e.g. ["S01F", "S01M", "S02F"] to split the work over Kaggle sessions
TRAIN_ALL_SPEAKERS_TOO = False   # also train one extra model on ALL speakers -> phase1_ALL.pt (for deployment, not for scoring)

P1_DIR = os.path.join(OUTPUT_DIR, "phase1_loso"); os.makedirs(P1_DIR, exist_ok=True)
P1_RESULTS = os.path.join(P1_DIR, "phase1_loso_results.json")

def safe(g): return re.sub(r"[^A-Za-z0-9_.-]", "_", g)

def compute_class_weights(train_sa):
    counts = torch.zeros(NUM_CLASSES)
    for s in train_sa:
        counts[s["hard_label"].argmax().item()] += 1
    w = counts.sum() / (NUM_CLASSES * counts.clamp(min=1))          # inverse class frequency
    if MAX_CLASS_WEIGHT is not None:
        w = w.clamp(max=MAX_CLASS_WEIGHT)
    return w.to(device)

class TemporalOnlyClassifier(nn.Module):
    """M1: Wav2Vec2.0 temporal module (Eq. 24-28) + linear head. The head is only used in Phase 1."""
    def __init__(self):
        super().__init__()
        self.temporal = TemporalWav2Vec2(freeze_transformer_layers=FREEZE_TRANSFORMER_LAYERS)
        self.head = nn.Linear(768, NUM_CLASSES)
    def forward(self, waveform, attention_mask=None):
        return self.head(self.temporal(waveform, attention_mask=attention_mask))

@torch.no_grad()
def predict(model, samples, bs=16):
    model.eval()
    loader = DataLoader(PTFileDataset(samples), batch_size=bs, shuffle=False, collate_fn=collate_fn_training)
    par = maybe_parallel(model)
    T, P = [], []
    for b in loader:
        logits = par(b["waveform"].to(device), attention_mask=b["attention_mask"].to(device))
        T += b["label"].argmax(-1).tolist(); P += logits.argmax(-1).cpu().tolist()
    return np.array(T), np.array(P)

def wa_ua(t, p):
    """WA = overall accuracy; UA = mean per-class recall over classes present in t."""
    if len(t) == 0: return float("nan"), float("nan")
    return float((t == p).mean() * 100), float(np.mean([(p[t == c] == c).mean() for c in np.unique(t)]) * 100)

def conf_mat(t, p):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for a, b in zip(t, p): cm[a, b] += 1
    return cm

In [9]:
# ============================================================================
# One Phase-1 run: train on `train_sa`, optionally score on `test_sa`, save phase1_<tag>.pt
# ============================================================================
def run_phase1(tag, train_sa, test_sa, test_group=None):
    final_path = os.path.join(P1_DIR, f"phase1_{safe(tag)}.pt")
    resume_path = os.path.join(P1_DIR, f"{safe(tag)}_resume.pt")
    cw = compute_class_weights(train_sa)
    print(f"\n{'='*78}\nPHASE 1 [{tag}]: train SA = {len(train_sa)}, test SA = {len(test_sa)}"
          f"\nclass weights: {dict(zip(EMOTION_CLASSES, [round(x, 2) for x in cw.tolist()]))}\n{'='*78}")

    model = TemporalOnlyClassifier().to(device)
    cfg = model.temporal.wav2vec2.config
    print(f"  masking (HF default): apply_spec_augment={cfg.apply_spec_augment}, mask_time_prob={cfg.mask_time_prob}, "
          f"mask_time_length={cfg.mask_time_length}")
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=PHASE1_LR)   # fixed LR
    start = 0
    if os.path.exists(resume_path):
        ck = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model_state"]); opt.load_state_dict(ck["optimizer_state"])
        start = ck["epoch"] + 1
        print(f"  RESUMING from epoch {start}/{EPOCHS_PHASE1}")

    par = maybe_parallel(model)
    loader = DataLoader(PTFileDataset(train_sa), batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate_fn_training, drop_last=True)
    n_skipped = 0
    t0 = time.time()
    for epoch in range(start, EPOCHS_PHASE1):
        model.train(); tot, nv = 0.0, 0
        for b in loader:
            opt.zero_grad()
            logits = par(b["waveform"].to(device), attention_mask=b["attention_mask"].to(device))
            loss = F.cross_entropy(logits, b["label"].to(device).argmax(-1), weight=cw)    # class-weighted CE
            if not torch.isfinite(loss):
                n_skipped += 1; continue
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(gn):
                n_skipped += 1; opt.zero_grad(); continue
            opt.step(); tot += loss.item(); nv += 1
        msg = (f"  [{tag}] epoch {epoch+1:3d}/{EPOCHS_PHASE1}  loss {tot/max(nv,1):.4f}  "
               f"(steps {nv}, skipped {n_skipped})  {(time.time()-t0)/(epoch-start+1)/60:.1f} min/epoch")
        if EVAL_EVERY and test_sa and (epoch + 1) % EVAL_EVERY == 0 and (epoch + 1) < EPOCHS_PHASE1:
            t, p = predict(model, test_sa); wa, ua = wa_ua(t, p)
            msg += f"  | monitor (held-out SA) WA {wa:.1f} UA {ua:.1f}"
        print(msg)
        if (epoch + 1) % CHECKPOINT_EVERY == 0 and (epoch + 1) < EPOCHS_PHASE1:
            torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": opt.state_dict()}, resume_path)

    res = {"tag": tag, "n_train_SA": len(train_sa), "n_test_SA": len(test_sa), "skipped_steps": n_skipped}
    if test_sa:                                                    # FINAL-epoch score on the held-out speaker
        t, p = predict(model, test_sa)
        wa, ua = wa_ua(t, p)
        res.update({"WA": wa, "UA": ua, "cm": conf_mat(t, p).tolist(),
                    "pred_dist": {EMOTION_CLASSES[k]: int(v) for k, v in zip(*np.unique(p, return_counts=True))}})
        print(f"  >> [{tag}] FINAL held-out SA:  WA {wa:.2f}   UA {ua:.2f}   "
              f"(classes predicted: {len(res['pred_dist'])}/{NUM_CLASSES})")

    torch.save({"temporal_state": {k: v.detach().cpu().clone() for k, v in model.temporal.state_dict().items()},
                "head_state": {k: v.detach().cpu().clone() for k, v in model.head.state_dict().items()},
                "current_frozen": FREEZE_TRANSFORMER_LAYERS,           # Phase 2/3 reads this key
                "epochs_trained": EPOCHS_PHASE1, "lr": PHASE1_LR,
                "test_group": test_group, "class_names": EMOTION_CLASSES,
                "train_groups": sorted({s["group"] for s in train_sa})}, final_path)
    res["minutes"] = round((time.time() - t0) / 60, 1)
    print(f"  saved -> {final_path}")
    if os.path.exists(resume_path): os.remove(resume_path)
    del model, par, opt; gc.collect(); torch.cuda.empty_cache()
    return res

In [10]:
# ============================================================================
# RUN LOSO Phase 1  (re-runnable: finished folds are skipped)
# ============================================================================
results = {}
if os.path.exists(P1_RESULTS):
    results = {r["tag"]: r for r in json.load(open(P1_RESULTS))}
    print("Already finished:", sorted(results))

todo = [g for g in (FOLDS_TO_RUN or GROUPS) if g not in results or
        not os.path.exists(os.path.join(P1_DIR, f"phase1_{safe(g)}.pt"))]
print("To run:", todo)

for g in todo:
    train_sa = [s for s in all_samples if s["group"] != g and s["split"] == "SA"]
    test_sa  = [s for s in all_samples if s["group"] == g and s["split"] == "SA"]
    assert not ({s["utt_id"] for s in train_sa} & {s["utt_id"] for s in test_sa}), "train/test overlap!"
    results[g] = run_phase1(g, train_sa, test_sa, test_group=g)
    json.dump(list(results.values()), open(P1_RESULTS, "w"), indent=1)

if TRAIN_ALL_SPEAKERS_TOO and not os.path.exists(os.path.join(P1_DIR, "phase1_ALL.pt")):
    run_phase1("ALL", [s for s in all_samples if s["split"] == "SA"], [], test_group=None)
print("\nDone with this run.")

To run: ['Ses01F', 'Ses01M', 'Ses02F', 'Ses02M', 'Ses03F', 'Ses03M', 'Ses04F', 'Ses04M', 'Ses05F', 'Ses05M']

PHASE 1 [Ses01F]: train SA = 7006, test SA = 713
class weights: {'anger': 1.09, 'happiness': 0.9, 'sadness': 1.38, 'neutral': 0.67, 'frustration': 0.59, 'fear': 8.0}


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: /kaggle/input/datasets/bdivyabalamurugan/wav2vec-model/wav2vec2_stored
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[TemporalWav2Vec2] Froze transformer layers 0--1 of 12; layers 0-11 + pooling head remain trainable.
[TemporalWav2Vec2] WARNING: masked_spec_embed loaded from the checkpoint is already corrupted (max abs=1.888e+38), BEFORE any fine-tuning -- this is a broken checkpoint file, not a training issue. Re-initializing it to HF's own default Uniform(0, 1) range instead of using the broken saved value.
[TemporalWav2Vec2] Froze masked_spec_embed (mask-token vector) to prevent it from diverging during fine-tuning.
  masking (HF default): apply_spec_augment=True, mask_time_prob=0.05, mask_time_length=10
  [Ses01F] epoch   1/10  loss 1.5912  (steps 875, skipped 0)  6.3 min/epoch
  [Ses01F] epoch   2/10  loss 1.3881  (steps 875, skipped 0)  6.3 min/epoch
  [Ses01F] epoch   3/10  loss 1.2743  (steps 875, skipped 0)  6.4 min/epoch
  [Ses01F] epoch   4/10  loss 1.1952  (steps 875, skipped 0)  6.4 min/epoch
  [Ses01F] epoch   5/10  loss 1.1179  (steps 875, skipped 0)  6.4 min/epoch  | monitor (held-out

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

[transformers] Wav2Vec2Model LOAD REPORT from: /kaggle/input/datasets/bdivyabalamurugan/wav2vec-model/wav2vec2_stored
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[TemporalWav2Vec2] Froze transformer layers 0--1 of 12; layers 0-11 + pooling head remain trainable.
[TemporalWav2Vec2] WARNING: masked_spec_embed loaded from the checkpoint is already corrupted (max abs=1.888e+38), BEFORE any fine-tuning -- this is a broken checkpoint file, not a training issue. Re-initializing it to HF's own default Uniform(0, 1) range instead of using the broken saved value.
[TemporalWav2Vec2] Froze masked_spec_embed (mask-token vector) to prevent it from diverging during fine-tuning.
  masking (HF default): apply_spec_augment=True, mask_time_prob=0.05, mask_time_length=10
  [Ses03F] epoch   1/10  loss 1.6327  (steps 886, skipped 0)  6.4 min/epoch
  [Ses03F] epoch   2/10  loss 1.4321  (steps 886, skipped 0)  6.4 min/epoch
  [Ses03F] epoch   3/10  loss 1.3043  (steps 886, skipped 0)  6.4 min/epoch
  [Ses03F] epoch   4/10  loss 1.2276  (steps 886, skipped 0)  6.4 min/epoch
  [Ses03F] epoch   5/10  loss 1.1496  (steps 886, skipped 0)  6.4 min/epoch  | monitor (held-out

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: /kaggle/input/datasets/bdivyabalamurugan/wav2vec-model/wav2vec2_stored
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[TemporalWav2Vec2] Froze transformer layers 0--1 of 12; layers 0-11 + pooling head remain trainable.
[TemporalWav2Vec2] WARNING: masked_spec_embed loaded from the checkpoint is already corrupted (max abs=1.888e+38), BEFORE any fine-tuning -- this is a broken checkpoint file, not a training issue. Re-initializing it to HF's own default Uniform(0, 1) range instead of using the broken saved value.
[TemporalWav2Vec2] Froze masked_spec_embed (mask-token vector) to prevent it from diverging during fine-tuning.
  masking (HF default): apply_spec_augment=True, mask_time_prob=0.05, mask_time_length=10
  [Ses03M] epoch   1/10  loss 1.6331  (steps 866, skipped 0)  6.3 min/epoch
  [Ses03M] epoch   2/10  loss 1.4541  (steps 866, skipped 0)  6.3 min/epoch
  [Ses03M] epoch   3/10  loss 1.3442  (steps 866, skipped 0)  6.3 min/epoch
  [Ses03M] epoch   4/10  loss 1.2555  (steps 866, skipped 0)  6.3 min/epoch
  [Ses03M] epoch   5/10  loss 1.1915  (steps 866, skipped 0)  6.3 min/epoch  | monitor (held-out

In [ ]:
# ============================================================================
# SUMMARY: Phase-1 (temporal-only) LOSO results on clear held-out utterances
# ============================================================================
from scipy import stats
import matplotlib.pyplot as plt

R = sorted(json.load(open(P1_RESULTS)), key=lambda r: r["tag"])
R = [r for r in R if "WA" in r]
print(f"{len(R)} folds finished\n")
print(f"{'held-out':10s}{'#SA':>6s}{'WA':>8s}{'UA':>8s}")
for r in R:
    print(f"{r['tag']:10s}{r['n_test_SA']:6d}{r['WA']:8.2f}{r['UA']:8.2f}")

def mean_ci(v):
    v = np.array([x for x in v if not math.isnan(x)]); n = len(v)
    if n < 2: return (v.mean() if n else float("nan")), float("nan"), float("nan"), float("nan"), n
    sd = v.std(ddof=1); h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)
    return v.mean(), sd, v.mean() - h, v.mean() + h, n

print()
for k in ("WA", "UA"):
    m, sd, lo, hi, n = mean_ci([r[k] for r in R])
    print(f"{k}: {m:.2f} +- {sd:.2f}   95% CI [{lo:.2f}, {hi:.2f}]   (n={n} folds)")

cm = np.sum([np.array(r["cm"]) for r in R], axis=0)
print("\nPooled confusion matrix (rows = true, cols = predicted):")
print(" " * 12 + "".join(f"{c[:8]:>10s}" for c in EMOTION_CLASSES) + f"{'recall':>9s}{'support':>9s}")
for i, row in enumerate(cm):
    print(f"{EMOTION_CLASSES[i]:12s}" + "".join(f"{v:10d}" for v in row) + f"{(row[i]/row.sum() if row.sum() else float('nan')):9.3f}{row.sum():9d}")

norm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(EMOTION_CLASSES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(EMOTION_CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Phase 1 LOSO (held-out speakers), row-normalised")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center", color="white" if norm[i, j] > .5 else "black")
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.savefig(os.path.join(P1_DIR, "phase1_loso_confusion.png"), dpi=150); plt.show()

## 6. Using these checkpoints in the Phase 2/3 LOSO notebook
Every fold's file is `phase1_<speaker>.pt` with the keys `temporal_state` and `current_frozen` (the same keys your old `phase1_final.pt` had). In `train_fold()` of the Phase 2/3 notebook, replace the Phase-1 block by:

```python
ck1 = torch.load(os.path.join(PHASE1_DIR, f"phase1_{test_spk}.pt"), map_location="cpu", weights_only=False)
assert ck1["test_group"] == test_spk          # never use a checkpoint that saw the test speaker
t1_state, phase1_frozen = ck1["temporal_state"], ck1["current_frozen"]
```
where `PHASE1_DIR` is the attached `phase1_loso` folder.